#### Step 5: Tokenize and pack the final training dataset

**Why:** everything before this step produced text (`dataset/sampled_10b/`, the 10B token, Hindi 40 / English 35 / Marathi 25 mix). Nothing on disk yet is actual `input_ids` a model can train on. This step:

1. Holds out **0.5%** of rows per language as validation, before any tokenization, so the model never sees them.
2. Tokenizes everything with the frozen `AxisQuant/IndicPrayog-tokenizer-64k` tokenizer.
3. Packs the token ids into fixed length rows of **2048 tokens** each (the model's trained context length), with an `<eos>` id inserted between documents, exactly like `final_plan.md`'s Phase 2 describes.
4. Writes the result as flat binary `.bin` files (`uint16`, since vocab size 64,000 fits in 16 bits): `train.bin` (all 3 languages, shard order shuffled so training batches are not one language at a time) and one `val_<language>.bin` per language, so per-language perplexity can be measured later without re-tokenizing anything.

**Known gap, recorded on purpose:** this run skips quality filtering and MinHash dedup (see the deviation note in `row_not_push_material/final_plan.md`). Whatever duplicate or low quality documents exist in `sampled_10b/` are packed as is.

In [2]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer

SAMPLED_ROOT = Path("../../../../dataset/sampled_10b").resolve()
OUTPUT_ROOT = Path("../../../../dataset/final_packed").resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

TOKENIZER_REPO = "AxisQuant/IndicPrayog-tokenizer-64k"
SEQ_LEN = 2048
VAL_FRACTION = 0.005
SEED = 42

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_REPO)
EOS_ID = tokenizer.eos_token_id
assert tokenizer.vocab_size <= 65536, "uint16 packing assumes the vocab fits in 16 bits"
print(f"eos token id: {EOS_ID}, vocab size: {tokenizer.vocab_size}")

eos token id: 1, vocab size: 64000


#### Find every sampled shard, shuffle the processing order

We shuffle the shard order (not the rows inside a shard) before packing the training set. That way `train.bin` interleaves languages and sources instead of being one huge Hindi block, then one huge English block, then Marathi. This is the "shuffle shards" step `final_plan.md` calls for.

In [3]:
shards = sorted(SAMPLED_ROOT.glob("*/*/*.parquet"))
print(f"found {len(shards)} shard(s) to pack")

shard_order = shards.copy()
random.Random(SEED).shuffle(shard_order)

found 149 shard(s) to pack


#### The sequence packer

A small buffer per output file: every document's token ids (plus one `<eos>`) get appended to the buffer, and whenever the buffer holds 2048 or more tokens, the first 2048 are cut off and written to disk as one row. This is how a handful of short documents end up sharing one training row, and how one very long document ends up spanning several rows. Whatever is left in the buffer at the very end (under 2048 tokens) is dropped, that is a normal, tiny amount of loss (at most 2047 tokens per file).

In [4]:
class SequencePacker:
    def __init__(self, path, seq_len, eos_id):
        self.seq_len = seq_len
        self.eos_id = eos_id
        self.buffer = []
        self.rows_written = 0
        self.file = open(path, "wb")

    def add(self, token_ids):
        self.buffer.extend(token_ids)
        self.buffer.append(self.eos_id)
        while len(self.buffer) >= self.seq_len:
            row = self.buffer[:self.seq_len]
            self.buffer = self.buffer[self.seq_len:]
            np.asarray(row, dtype=np.uint16).tofile(self.file)
            self.rows_written += 1

    def close(self):
        dropped = len(self.buffer)
        self.file.close()
        return dropped

#### Run: split held out validation, tokenize, pack

For every shard: draw a per row val/train mask (seeded, so reproducible), tokenize every row once, and feed each row's token ids into the right packer, `train_packer` or `val_packers[language]`. A real benchmark on this tokenizer measured about 4,160 docs/sec, `sampled_10b` has roughly 16 million rows, so expect this cell to take **roughly 65 minutes**.

In [5]:
train_packer = SequencePacker(OUTPUT_ROOT / "train.bin", SEQ_LEN, EOS_ID)
val_packers = {
    language: SequencePacker(OUTPUT_ROOT / f"val_{language}.bin", SEQ_LEN, EOS_ID)
    for language in ["en", "hi", "mr"]
}

val_rng = np.random.RandomState(SEED)
rows_seen = {"train": 0, "val_en": 0, "val_hi": 0, "val_mr": 0}

shard_bar = tqdm(shard_order, desc="shards", unit="shard")
for shard in shard_bar:
    language = shard.parent.parent.name
    source = shard.parent.name
    shard_bar.set_postfix(current=f"{language}/{source}/{shard.name}")

    df = pd.read_parquet(shard, columns=["text"])
    texts = df["text"].tolist()
    is_val = val_rng.random_sample(len(texts)) < VAL_FRACTION

    encoded = tokenizer(texts)["input_ids"]
    for token_ids, val_flag in zip(encoded, is_val):
        if val_flag:
            val_packers[language].add(token_ids)
            rows_seen[f"val_{language}"] += 1
        else:
            train_packer.add(token_ids)
            rows_seen["train"] += 1

dropped_tail_tokens = {"train": train_packer.close()}
for language, packer in val_packers.items():
    dropped_tail_tokens[f"val_{language}"] = packer.close()

print(f"documents packed into train: {rows_seen['train']:,}")
for language in ["en", "hi", "mr"]:
    print(f"documents packed into val_{language}: {rows_seen[f'val_{language}']:,}")

shards: 100%|██████████| 149/149 [59:11<00:00, 23.83s/shard, current=hi/sangraha/data-23.parquet]              

documents packed into train: 15,993,301
documents packed into val_en: 17,951
documents packed into val_hi: 36,304
documents packed into val_mr: 26,534


#### Final report: rows and tokens actually written

In [6]:
report_rows = []
for name, packer in [("train", train_packer), *[(f"val_{l}", p) for l, p in val_packers.items()]]:
    report_rows.append({
        "split": name,
        "rows_written": packer.rows_written,
        "tokens_written": packer.rows_written * SEQ_LEN,
        "tail_tokens_dropped": dropped_tail_tokens[name],
    })

report = pd.DataFrame(report_rows)
report.to_csv(OUTPUT_ROOT / "final_pack_report.csv", index=False)

train_tokens = report.loc[report["split"] == "train", "tokens_written"].iloc[0]
print(f"train tokens: {train_tokens:,}")
print(f"saved: {(OUTPUT_ROOT / 'final_pack_report.csv').resolve()}")
report

train tokens: 9,960,701,952
saved: /home/contributor/users/prashant.takale/github_v2/dataset/final_packed/final_pack_report.csv


,split,rows_written,tokens_written,tail_tokens_dropped
0,train,4863624,9960701952,597
1,val_en,8601,17614848,1189
2,val_hi,9890,20254720,1672
3,val_mr,6224,12746752,1244
